# Predição da produtividade leiteira municipal com aprendizado de máquina

Este notebook prepara os dados anuais do SIDRA/IBGE, constrói pares município-ano para previsão da produtividade leiteira no ano seguinte e compara cinco abordagens:

- persistência;
- regressão Ridge;
- Random Forest;
- XGBoost;
- rede neural Multilayer Perceptron (MLP).

São avaliados três conjuntos progressivos de informações: histórico, pecuário e integrado. A divisão é estritamente temporal: treinamento em 2000–2013, validação em 2014–2018 e teste em 2019–2023, sempre utilizando as características do ano atual para estimar a produtividade do ano seguinte.

O notebook foi estruturado para execução integral no Google Colab. O arquivo `jsons_ibge_baixados.zip` deve estar no Google Drive, preferencialmente na mesma pasta em que o notebook foi armazenado. Os resultados são gravados em uma subpasta criada ao lado do arquivo de dados.

## 1. Ambiente computacional

In [ ]:
# Instala apenas os pacotes ausentes no ambiente.
import importlib.util
import subprocess
import sys

PACOTES = {
    "xgboost": "xgboost",
    "tensorflow": "tensorflow",
    "openpyxl": "openpyxl",
    "pyarrow": "pyarrow",
}

for modulo, pacote in PACOTES.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])

In [ ]:
import gc
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from collections import Counter
from itertools import product
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import xgboost as xgb
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers, regularizers

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SEMENTE = 42
os.environ["PYTHONHASHSEED"] = str(SEMENTE)
random.seed(SEMENTE)
np.random.seed(SEMENTE)
tf.keras.utils.set_random_seed(SEMENTE)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("TensorFlow:", tf.__version__)
print("GPU disponível:", bool(tf.config.list_physical_devices("GPU")))

## 2. Localização automática dos dados

In [ ]:
# Monta o Google Drive quando o notebook é executado no Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

NOME_ARQUIVO_DADOS = "jsons_ibge_baixados.zip"


def localizar_arquivo_dados(nome_arquivo: str) -> Path:
    """Procura o arquivo no diretório local e no Google Drive."""
    raizes = [Path("/content"), Path.cwd()]
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        raizes.append(drive_root)

    encontrados = []
    vistos = set()
    for raiz in raizes:
        if not raiz.exists():
            continue
        candidato_direto = raiz / nome_arquivo
        if candidato_direto.exists():
            encontrados.append(candidato_direto)
        try:
            for caminho in raiz.rglob(nome_arquivo):
                chave = str(caminho.resolve())
                if chave not in vistos:
                    vistos.add(chave)
                    encontrados.append(caminho)
        except (OSError, PermissionError):
            continue

    if not encontrados:
        raise FileNotFoundError(
            f"O arquivo '{nome_arquivo}' não foi localizado em /content nem no Google Drive."
        )

    # Prioriza o arquivo mais recentemente modificado quando houver cópias.
    encontrados = sorted(set(encontrados), key=lambda p: p.stat().st_mtime, reverse=True)
    if len(encontrados) > 1:
        print("Foram encontradas várias cópias. Será utilizada:", encontrados[0])
    return encontrados[0]


ARQUIVO_DADOS = localizar_arquivo_dados(NOME_ARQUIVO_DADOS)
PASTA_PROJETO = ARQUIVO_DADOS.parent
PASTA_RESULTADOS = PASTA_PROJETO / "resultados_produtividade_leiteira_ml"
PASTA_FIGURAS = PASTA_RESULTADOS / "figuras"
PASTA_MODELOS = PASTA_RESULTADOS / "modelos"

PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

print("Arquivo de dados:", ARQUIVO_DADOS)
print("Pasta de resultados:", PASTA_RESULTADOS)

## 3. Parâmetros do experimento

As grades foram mantidas deliberadamente compactas para permitir a execução integral no Colab. A seleção é feita exclusivamente pelo conjunto de validação. O conjunto de teste permanece isolado até a avaliação final.

In [ ]:
ANOS = list(range(2000, 2025))

INTERVALOS = {
    "treino": (2000, 2013),
    "validacao": (2014, 2018),
    "teste": (2019, 2023),
}

GRADE_RIDGE = [0.01, 0.1, 1.0, 10.0, 100.0]

GRADE_RF = [
    {"n_estimators": 250, "max_depth": 12, "min_samples_leaf": 1, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": 20, "min_samples_leaf": 1, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": None, "min_samples_leaf": 2, "max_features": 0.7},
    {"n_estimators": 250, "max_depth": None, "min_samples_leaf": 4, "max_features": 1.0},
]

GRADE_XGB = [
    {"max_depth": 4, "learning_rate": 0.03},
    {"max_depth": 4, "learning_rate": 0.08},
    {"max_depth": 7, "learning_rate": 0.03},
    {"max_depth": 7, "learning_rate": 0.08},
]

GRADE_MLP = [
    {"camadas": (32, 16), "dropout": 0.10, "taxa_aprendizado": 1e-3},
    {"camadas": (64, 32), "dropout": 0.10, "taxa_aprendizado": 1e-3},
]

MAX_EPOCAS_MLP = 200
PACIENCIA_MLP = 20
TAMANHO_LOTE = 512
PACIENCIA_XGB = 40
MAX_ARVORES_XGB = 1200

# Pontos temporais usados nas curvas de aprendizado de Ridge e Random Forest.
CORTES_CURVA_APRENDIZADO = [2003, 2006, 2009, 2011, 2013]

## 4. Leitura e integração das tabelas do SIDRA/IBGE

Convenções aplicadas aos símbolos especiais do SIDRA:

- `-`: zero absoluto;
- `0`: zero numérico;
- `X`, `..`, `...` e campos vazios: valor indisponível, não aplicável ou inibido, mantido como ausente.

A produção de leite da Tabela 74 é disponibilizada em mil litros e, por isso, é multiplicada por 1.000. As demais unidades são preservadas conforme os arquivos de origem.

In [ ]:
SIMBOLOS_ZERO = {"-"}
SIMBOLOS_AUSENTES = {"X", "x", "..", "...", "", None}
CONTAGEM_SIMBOLOS = Counter()


def converter_valor_sidra(valor):
    """Converte valores do SIDRA, preservando a distinção entre zero e ausência."""
    CONTAGEM_SIMBOLOS[str(valor)] += 1
    if valor in SIMBOLOS_ZERO:
        return 0.0
    if valor in SIMBOLOS_AUSENTES:
        return np.nan
    try:
        return float(str(valor).replace(",", "."))
    except (TypeError, ValueError):
        return np.nan


def localizar_membro_zip(zf: zipfile.ZipFile, expressao: str) -> str:
    encontrados = [nome for nome in zf.namelist() if re.search(expressao, nome)]
    if len(encontrados) != 1:
        raise ValueError(
            f"Esperado um arquivo para o padrão '{expressao}', mas foram encontrados {len(encontrados)}."
        )
    return encontrados[0]


def ler_json_zip(zf: zipfile.ZipFile, expressao: str):
    membro = localizar_membro_zip(zf, expressao)
    return json.loads(zf.read(membro).decode("utf-8"))


def extrair_item_sidra(item, ano, nome_coluna, fator=1.0, cultura=None):
    """Transforma um item da API SIDRA em uma tabela município-ano."""
    registros = []
    for resultado in item.get("resultados", []):
        categorias = {
            codigo: nome
            for classificacao in resultado.get("classificacoes", [])
            for codigo, nome in classificacao.get("categoria", {}).items()
        }
        if cultura is not None and cultura not in categorias.values():
            continue

        for serie in resultado.get("series", []):
            localidade = serie["localidade"]
            nome_com_uf = localidade.get("nome", "")
            correspondencia_uf = re.search(r" - ([A-Z]{2})$", nome_com_uf)
            valor = converter_valor_sidra(serie.get("serie", {}).get(str(ano)))
            registros.append(
                {
                    "codigo_municipio": str(localidade["id"]),
                    "municipio_uf": nome_com_uf,
                    "uf": correspondencia_uf.group(1) if correspondencia_uf else np.nan,
                    "ano": int(ano),
                    nome_coluna: valor * fator if pd.notna(valor) else np.nan,
                }
            )

    return pd.DataFrame(registros)


def carregar_ano(zf: zipfile.ZipFile, ano: int) -> pd.DataFrame:
    """Lê as quatro tabelas e devolve uma linha por município no ano informado."""
    tabela_74 = ler_json_zip(zf, rf"tabela_74/tabela_74_{ano}_.*\.json$")[0]
    tabela_94 = ler_json_zip(zf, rf"tabela_94/tabela_94_{ano}_.*\.json$")[0]
    tabela_3939 = ler_json_zip(zf, rf"tabela_3939/tabela_3939_{ano}_.*\.json$")[0]
    tabela_5457 = ler_json_zip(zf, rf"tabela_5457/tabela_5457_{ano}_.*\.json$")

    partes = [
        extrair_item_sidra(tabela_74, ano, "leite_litros", fator=1000.0),
        extrair_item_sidra(tabela_94, ano, "vacas_ordenhadas"),
        extrair_item_sidra(tabela_3939, ano, "bovinos"),
    ]

    nomes_variaveis_agricolas = {
        "8331": "area_plantada_ha",
        "216": "area_colhida_ha",
        "214": "quantidade_t",
        "112": "rendimento_kg_ha",
    }
    culturas = {
        "Milho (em grão)": "milho",
        "Soja (em grão)": "soja",
    }

    for item in tabela_5457:
        sufixo = nomes_variaveis_agricolas[item["id"]]
        for nome_sidra, prefixo in culturas.items():
            partes.append(
                extrair_item_sidra(
                    item,
                    ano,
                    f"{prefixo}_{sufixo}",
                    cultura=nome_sidra,
                )
            )

    chaves = ["codigo_municipio", "municipio_uf", "uf", "ano"]
    base_ano = partes[0]
    for parte in partes[1:]:
        base_ano = base_ano.merge(parte, on=chaves, how="outer", validate="one_to_one")

    return base_ano

In [ ]:
inicio = time.time()
partes_anuais = []

with zipfile.ZipFile(ARQUIVO_DADOS) as zf:
    for ano in ANOS:
        parte = carregar_ano(zf, ano)
        partes_anuais.append(parte)
        print(f"{ano}: {len(parte):,} municípios")

base_bruta = pd.concat(partes_anuais, ignore_index=True)
base_bruta["municipio"] = base_bruta["municipio_uf"].str.replace(
    r" - [A-Z]{2}$", "", regex=True
)

print(f"\nBase integrada: {base_bruta.shape[0]:,} linhas e {base_bruta.shape[1]} colunas")
print(f"Tempo de leitura: {(time.time() - inicio) / 60:.2f} minutos")

## 5. Verificações de consistência e preparação do painel

In [ ]:
CHAVES_PAINEL = ["codigo_municipio", "ano"]
COLUNAS_NUMERICAS_ORIGINAIS = [
    "leite_litros",
    "vacas_ordenhadas",
    "bovinos",
    "milho_area_plantada_ha",
    "milho_area_colhida_ha",
    "milho_quantidade_t",
    "milho_rendimento_kg_ha",
    "soja_area_plantada_ha",
    "soja_area_colhida_ha",
    "soja_quantidade_t",
    "soja_rendimento_kg_ha",
]

if base_bruta.duplicated(CHAVES_PAINEL).any():
    duplicados = base_bruta.loc[base_bruta.duplicated(CHAVES_PAINEL, keep=False)]
    raise ValueError(f"Foram encontradas {len(duplicados)} linhas duplicadas no painel.")

negativos = (base_bruta[COLUNAS_NUMERICAS_ORIGINAIS] < 0).sum().sum()
if negativos > 0:
    raise ValueError(f"Foram encontrados {int(negativos)} valores negativos inesperados.")

base_bruta["produtividade_l_vaca_ano"] = np.where(
    base_bruta["vacas_ordenhadas"] > 0,
    base_bruta["leite_litros"] / base_bruta["vacas_ordenhadas"],
    np.nan,
)

inconsistencia_rebanho = base_bruta.loc[
    base_bruta["vacas_ordenhadas"].notna()
    & base_bruta["bovinos"].notna()
    & (base_bruta["vacas_ordenhadas"] > base_bruta["bovinos"]),
    ["codigo_municipio", "municipio_uf", "ano", "vacas_ordenhadas", "bovinos"],
].copy()

resumo_qualidade = pd.DataFrame(
    {
        "indicador": [
            "Linhas do painel",
            "Duplicidades município-ano",
            "Produtividades válidas",
            "Vacas ordenhadas superiores ao rebanho bovino",
        ],
        "valor": [
            len(base_bruta),
            int(base_bruta.duplicated(CHAVES_PAINEL).sum()),
            int(base_bruta["produtividade_l_vaca_ano"].notna().sum()),
            len(inconsistencia_rebanho),
        ],
    }
)

display(resumo_qualidade)
display(base_bruta[COLUNAS_NUMERICAS_ORIGINAIS + ["produtividade_l_vaca_ano"]].describe().T)

if not inconsistencia_rebanho.empty:
    print("Registros mantidos e sinalizados para inspeção:")
    display(inconsistencia_rebanho.head(20))

In [ ]:
# Exporta as verificações antes da modelagem.
resumo_qualidade.to_csv(PASTA_RESULTADOS / "resumo_qualidade_dados.csv", index=False)
inconsistencia_rebanho.to_csv(
    PASTA_RESULTADOS / "registros_vacas_superiores_rebanho.csv", index=False
)

simbolos_relevantes = pd.DataFrame(
    [
        {"simbolo": simbolo, "frequencia": frequencia}
        for simbolo, frequencia in CONTAGEM_SIMBOLOS.items()
        if simbolo in {"-", "0", "X", "x", "..", "...", "", "None"}
    ]
).sort_values("frequencia", ascending=False)
simbolos_relevantes.to_csv(PASTA_RESULTADOS / "contagem_simbolos_sidra.csv", index=False)
display(simbolos_relevantes)

## 6. Engenharia de atributos e construção do alvo

O conjunto integrado contém exatamente 21 variáveis explicativas. As variações são calculadas por diferenças de `ln(1 + X)`, o que permite tratar valores iguais a zero sem criar taxas percentuais indefinidas. As defasagens somente são aceitas quando os anos são consecutivos para o mesmo código municipal.

In [ ]:
base = base_bruta.sort_values(["codigo_municipio", "ano"]).reset_index(drop=True)
agrupado = base.groupby("codigo_municipio", sort=False)


def defasagem_consecutiva(coluna: str, passos: int = 1) -> pd.Series:
    valor_defasado = agrupado[coluna].shift(passos)
    ano_defasado = agrupado["ano"].shift(passos)
    return valor_defasado.where(ano_defasado.eq(base["ano"] - passos))


def variacao_logaritmica(coluna: str) -> pd.Series:
    anterior = defasagem_consecutiva(coluna, 1)
    return np.log1p(base[coluna]) - np.log1p(anterior)


# Histórico da produtividade.
prod_lag1 = defasagem_consecutiva("produtividade_l_vaca_ano", 1)
prod_lag2 = defasagem_consecutiva("produtividade_l_vaca_ano", 2)
base["variacao_log_produtividade_1a"] = (
    np.log1p(base["produtividade_l_vaca_ano"]) - np.log1p(prod_lag1)
)
base["media_produtividade_3a"] = pd.concat(
    [base["produtividade_l_vaca_ano"], prod_lag1, prod_lag2], axis=1
).mean(axis=1, skipna=False)

# Estrutura pecuária.
base["participacao_vacas_rebanho"] = np.where(
    base["bovinos"] > 0,
    base["vacas_ordenhadas"] / base["bovinos"],
    np.nan,
)
base["variacao_log_leite_1a"] = variacao_logaritmica("leite_litros")
base["variacao_log_vacas_1a"] = variacao_logaritmica("vacas_ordenhadas")
base["variacao_log_bovinos_1a"] = variacao_logaritmica("bovinos")

# Contexto agrícola.
base["variacao_log_milho_quantidade_1a"] = variacao_logaritmica("milho_quantidade_t")
base["variacao_log_soja_quantidade_1a"] = variacao_logaritmica("soja_quantidade_t")
base["graos_total_t"] = base[["milho_quantidade_t", "soja_quantidade_t"]].sum(
    axis=1, min_count=2
)
base["graos_kg_por_vaca"] = np.where(
    base["vacas_ordenhadas"] > 0,
    1000.0 * base["graos_total_t"] / base["vacas_ordenhadas"],
    np.nan,
)

# A produtividade do ano seguinte é o alvo da regressão.
base["ano_seguinte"] = agrupado["ano"].shift(-1)
base["produtividade_alvo"] = agrupado["produtividade_l_vaca_ano"].shift(-1)
base["produtividade_alvo"] = base["produtividade_alvo"].where(
    base["ano_seguinte"].eq(base["ano"] + 1)
)
base["ano_alvo"] = base["ano"] + 1

In [ ]:
VARIAVEIS_HISTORICAS = [
    "produtividade_l_vaca_ano",
    "variacao_log_produtividade_1a",
    "media_produtividade_3a",
]

VARIAVEIS_PECUARIAS_ADICIONAIS = [
    "leite_litros",
    "vacas_ordenhadas",
    "bovinos",
    "participacao_vacas_rebanho",
    "variacao_log_leite_1a",
    "variacao_log_vacas_1a",
    "variacao_log_bovinos_1a",
]

VARIAVEIS_AGRICOLAS_ADICIONAIS = [
    "milho_area_plantada_ha",
    "milho_area_colhida_ha",
    "milho_quantidade_t",
    "milho_rendimento_kg_ha",
    "soja_area_plantada_ha",
    "soja_area_colhida_ha",
    "soja_quantidade_t",
    "soja_rendimento_kg_ha",
    "variacao_log_milho_quantidade_1a",
    "variacao_log_soja_quantidade_1a",
    "graos_kg_por_vaca",
]

CONJUNTOS_VARIAVEIS = {
    "historico": VARIAVEIS_HISTORICAS,
    "pecuario": VARIAVEIS_HISTORICAS + VARIAVEIS_PECUARIAS_ADICIONAIS,
    "integrado": (
        VARIAVEIS_HISTORICAS
        + VARIAVEIS_PECUARIAS_ADICIONAIS
        + VARIAVEIS_AGRICOLAS_ADICIONAIS
    ),
}

for nome, variaveis in CONJUNTOS_VARIAVEIS.items():
    print(f"{nome}: {len(variaveis)} variáveis")

assert len(CONJUNTOS_VARIAVEIS["integrado"]) == 21

## 7. Base de modelagem e divisão cronológica

In [ ]:
COLUNAS_IDENTIFICACAO = [
    "codigo_municipio",
    "municipio",
    "municipio_uf",
    "uf",
    "ano",
    "ano_alvo",
]

base_modelagem = base.loc[
    base["produtividade_l_vaca_ano"].notna()
    & base["produtividade_alvo"].notna()
].copy()

REGIOES = {
    "Norte": ["AC", "AP", "AM", "PA", "RO", "RR", "TO"],
    "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
    "Centro-Oeste": ["DF", "GO", "MT", "MS"],
    "Sudeste": ["ES", "MG", "RJ", "SP"],
    "Sul": ["PR", "RS", "SC"],
}
MAPA_REGIAO = {uf: regiao for regiao, ufs in REGIOES.items() for uf in ufs}
base_modelagem["regiao"] = base_modelagem["uf"].map(MAPA_REGIAO)


def selecionar_periodo(df, inicio, fim):
    return df.loc[df["ano"].between(inicio, fim)].copy()


treino = selecionar_periodo(base_modelagem, *INTERVALOS["treino"])
validacao = selecionar_periodo(base_modelagem, *INTERVALOS["validacao"])
teste = selecionar_periodo(base_modelagem, *INTERVALOS["teste"])

assert treino["ano"].max() < validacao["ano"].min()
assert validacao["ano"].max() < teste["ano"].min()

resumo_amostras = pd.DataFrame(
    [
        {
            "conjunto": nome,
            "ano_caracteristicas": f"{df['ano'].min()}–{df['ano'].max()}",
            "ano_alvo": f"{df['ano_alvo'].min()}–{df['ano_alvo'].max()}",
            "observacoes": len(df),
            "municipios": df["codigo_municipio"].nunique(),
        }
        for nome, df in [("Treino", treino), ("Validação", validacao), ("Teste", teste)]
    ]
)
display(resumo_amostras)
resumo_amostras.to_csv(PASTA_RESULTADOS / "resumo_amostras.csv", index=False)

In [ ]:
# Descrição do alvo por período.
resumo_alvo = (
    pd.concat(
        [
            treino.assign(conjunto="Treino"),
            validacao.assign(conjunto="Validação"),
            teste.assign(conjunto="Teste"),
        ]
    )
    .groupby("conjunto")["produtividade_alvo"]
    .describe()
)
display(resumo_alvo)
resumo_alvo.to_csv(PASTA_RESULTADOS / "resumo_produtividade_alvo.csv")

plt.figure(figsize=(8, 5))
for nome, df in [("Treino", treino), ("Validação", validacao), ("Teste", teste)]:
    plt.hist(df["produtividade_alvo"], bins=50, alpha=0.35, label=nome, density=True)
plt.xlabel("Produtividade no ano seguinte (L/vaca/ano)")
plt.ylabel("Densidade")
plt.title("Distribuição da variável-alvo")
plt.legend()
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "distribuicao_alvo.png", dpi=300)
plt.show()

## 8. Funções de avaliação e modelos

O ajuste de valores ausentes e a padronização, quando necessária, são aprendidos somente com os dados de treinamento. Na MLP, a variável-alvo também é padronizada durante o ajuste e depois reconvertida para a unidade original. O MAE e o RMSE permanecem expressos em litros por vaca ao ano.

In [ ]:
ALVO = "produtividade_alvo"


def calcular_metricas(y_real, y_predito):
    return {
        "MAE": mean_absolute_error(y_real, y_predito),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_predito)),
        "R2": r2_score(y_real, y_predito),
    }


def criar_pipeline_ridge(alpha):
    return Pipeline(
        [
            ("imputacao", SimpleImputer(strategy="median")),
            ("padronizacao", StandardScaler()),
            ("modelo", Ridge(alpha=alpha)),
        ]
    )


def criar_pipeline_rf(parametros):
    return Pipeline(
        [
            ("imputacao", SimpleImputer(strategy="median")),
            (
                "modelo",
                RandomForestRegressor(
                    **parametros,
                    random_state=SEMENTE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def criar_xgb(parametros, n_estimators=MAX_ARVORES_XGB, early_stopping=True):
    argumentos = dict(
        n_estimators=n_estimators,
        objective="reg:squarederror",
        eval_metric="rmse",
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=1.0,
        tree_method="hist",
        random_state=SEMENTE,
        n_jobs=-1,
        **parametros,
    )
    if early_stopping:
        argumentos["early_stopping_rounds"] = PACIENCIA_XGB
    return xgb.XGBRegressor(**argumentos)


def construir_mlp(n_entradas, camadas_ocultas, dropout, taxa_aprendizado):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEMENTE)

    modelo = keras.Sequential(name="mlp_produtividade_leiteira")
    modelo.add(layers.Input(shape=(n_entradas,)))
    for unidades in camadas_ocultas:
        modelo.add(
            layers.Dense(
                unidades,
                activation="relu",
                kernel_regularizer=regularizers.l2(1e-5),
            )
        )
        if dropout > 0:
            modelo.add(layers.Dropout(dropout))
    modelo.add(layers.Dense(1, activation="linear"))

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=taxa_aprendizado),
        loss="mse",
        metrics=[
            keras.metrics.RootMeanSquaredError(name="rmse"),
            keras.metrics.MeanAbsoluteError(name="mae"),
        ],
    )
    return modelo

## 9. Seleção de hiperparâmetros no conjunto de validação

In [ ]:
resultados_validacao = []
melhores_parametros = {}
objetos_validacao = {}

# Persistência: a previsão do próximo ano é a produtividade observada no ano atual.
pred_validacao_persistencia = validacao["produtividade_l_vaca_ano"].to_numpy()
metricas_persistencia_validacao = calcular_metricas(
    validacao[ALVO], pred_validacao_persistencia
)
resultados_validacao.append(
    {
        "configuracao": "referencia",
        "modelo": "Persistência",
        **metricas_persistencia_validacao,
        "parametros": "produtividade(t+1) = produtividade(t)",
    }
)

for configuracao, variaveis in CONJUNTOS_VARIAVEIS.items():
    print(f"\n=== Configuração: {configuracao} ({len(variaveis)} variáveis) ===")
    X_treino = treino[variaveis]
    y_treino = treino[ALVO]
    X_validacao = validacao[variaveis]
    y_validacao = validacao[ALVO]

    # Ridge
    candidatos = []
    for alpha in GRADE_RIDGE:
        modelo = criar_pipeline_ridge(alpha)
        modelo.fit(X_treino, y_treino)
        pred = modelo.predict(X_validacao)
        metricas = calcular_metricas(y_validacao, pred)
        candidatos.append((metricas["RMSE"], alpha, modelo, metricas))
    _, alpha, modelo, metricas = min(candidatos, key=lambda item: item[0])
    melhores_parametros[(configuracao, "Ridge")] = {"alpha": alpha}
    objetos_validacao[(configuracao, "Ridge")] = modelo
    resultados_validacao.append(
        {
            "configuracao": configuracao,
            "modelo": "Ridge",
            **metricas,
            "parametros": json.dumps({"alpha": alpha}),
        }
    )
    print("Ridge concluída.")

    # Random Forest
    candidatos = []
    for parametros in GRADE_RF:
        modelo = criar_pipeline_rf(parametros)
        modelo.fit(X_treino, y_treino)
        pred = modelo.predict(X_validacao)
        metricas = calcular_metricas(y_validacao, pred)
        candidatos.append((metricas["RMSE"], parametros, modelo, metricas))
    _, parametros, modelo, metricas = min(candidatos, key=lambda item: item[0])
    melhores_parametros[(configuracao, "Random Forest")] = parametros
    objetos_validacao[(configuracao, "Random Forest")] = modelo
    resultados_validacao.append(
        {
            "configuracao": configuracao,
            "modelo": "Random Forest",
            **metricas,
            "parametros": json.dumps(parametros),
        }
    )
    print("Random Forest concluída.")

    # XGBoost
    imputador_xgb = SimpleImputer(strategy="median")
    X_treino_xgb = imputador_xgb.fit_transform(X_treino)
    X_validacao_xgb = imputador_xgb.transform(X_validacao)
    candidatos = []
    for parametros in GRADE_XGB:
        modelo = criar_xgb(parametros, early_stopping=True)
        modelo.fit(
            X_treino_xgb,
            y_treino,
            eval_set=[(X_treino_xgb, y_treino), (X_validacao_xgb, y_validacao)],
            verbose=False,
        )
        pred = modelo.predict(X_validacao_xgb)
        metricas = calcular_metricas(y_validacao, pred)
        candidatos.append((metricas["RMSE"], parametros, modelo, metricas))
    _, parametros, modelo, metricas = min(candidatos, key=lambda item: item[0])
    melhor_iteracao = int(getattr(modelo, "best_iteration", modelo.n_estimators - 1)) + 1
    melhores_parametros[(configuracao, "XGBoost")] = {
        **parametros,
        "n_estimators": melhor_iteracao,
    }
    objetos_validacao[(configuracao, "XGBoost")] = {
        "imputador": imputador_xgb,
        "modelo": modelo,
    }
    resultados_validacao.append(
        {
            "configuracao": configuracao,
            "modelo": "XGBoost",
            **metricas,
            "parametros": json.dumps(
                {**parametros, "n_estimators": melhor_iteracao}
            ),
        }
    )
    print("XGBoost concluído.")

    # MLP
    imputador_mlp = SimpleImputer(strategy="median")
    padronizador_mlp = StandardScaler()
    padronizador_alvo_mlp = StandardScaler()
    X_treino_mlp = padronizador_mlp.fit_transform(imputador_mlp.fit_transform(X_treino))
    X_validacao_mlp = padronizador_mlp.transform(imputador_mlp.transform(X_validacao))
    y_treino_mlp = padronizador_alvo_mlp.fit_transform(y_treino.to_numpy().reshape(-1, 1)).ravel()
    y_validacao_mlp = padronizador_alvo_mlp.transform(y_validacao.to_numpy().reshape(-1, 1)).ravel()
    candidatos = []

    for parametros in GRADE_MLP:
        modelo = construir_mlp(
            n_entradas=X_treino_mlp.shape[1],
            camadas_ocultas=parametros["camadas"],
            dropout=parametros["dropout"],
            taxa_aprendizado=parametros["taxa_aprendizado"],
        )
        parada = keras.callbacks.EarlyStopping(
            monitor="val_rmse",
            patience=PACIENCIA_MLP,
            restore_best_weights=True,
        )
        historico = modelo.fit(
            X_treino_mlp,
            y_treino_mlp,
            validation_data=(X_validacao_mlp, y_validacao_mlp),
            epochs=MAX_EPOCAS_MLP,
            batch_size=TAMANHO_LOTE,
            callbacks=[parada],
            verbose=0,
            shuffle=True,
        )
        pred_padronizada = modelo.predict(X_validacao_mlp, verbose=0).ravel()
        pred = padronizador_alvo_mlp.inverse_transform(pred_padronizada.reshape(-1, 1)).ravel()
        metricas = calcular_metricas(y_validacao, pred)
        candidatos.append((metricas["RMSE"], parametros, modelo, historico, metricas))

    _, parametros, modelo, historico, metricas = min(candidatos, key=lambda item: item[0])
    melhor_epoca = int(np.argmin(historico.history["val_rmse"])) + 1
    melhores_parametros[(configuracao, "MLP")] = {
        **parametros,
        "epocas": melhor_epoca,
    }
    objetos_validacao[(configuracao, "MLP")] = {
        "imputador": imputador_mlp,
        "padronizador": padronizador_mlp,
        "padronizador_alvo": padronizador_alvo_mlp,
        "modelo": modelo,
        "historico": historico.history,
    }
    resultados_validacao.append(
        {
            "configuracao": configuracao,
            "modelo": "MLP",
            **metricas,
            "parametros": json.dumps(
                {
                    "camadas": list(parametros["camadas"]),
                    "dropout": parametros["dropout"],
                    "taxa_aprendizado": parametros["taxa_aprendizado"],
                    "epocas": melhor_epoca,
                }
            ),
        }
    )
    print("MLP concluída.")

    gc.collect()

resultados_validacao = pd.DataFrame(resultados_validacao).sort_values("RMSE")
display(resultados_validacao)
resultados_validacao.to_csv(PASTA_RESULTADOS / "resultados_validacao.csv", index=False)

## 10. Curvas de treinamento e validação

Para Ridge e Random Forest, que não utilizam épocas, são calculadas curvas de aprendizado com janelas temporais crescentes de treinamento. Para XGBoost e MLP, são apresentadas as curvas nativas ao longo das rodadas de boosting e das épocas. A persistência é exibida como referência, embora não possua parâmetros ajustáveis.

In [ ]:
def gerar_curva_aprendizado_sklearn(configuracao, nome_modelo):
    variaveis = CONJUNTOS_VARIAVEIS[configuracao]
    linhas = []

    for ano_final in CORTES_CURVA_APRENDIZADO:
        subtreino = treino.loc[treino["ano"] <= ano_final]
        X_sub = subtreino[variaveis]
        y_sub = subtreino[ALVO]

        if nome_modelo == "Ridge":
            alpha = melhores_parametros[(configuracao, "Ridge")]["alpha"]
            modelo = criar_pipeline_ridge(alpha)
        elif nome_modelo == "Random Forest":
            parametros = melhores_parametros[(configuracao, "Random Forest")]
            modelo = criar_pipeline_rf(parametros)
        else:
            raise ValueError("Modelo não suportado nesta função.")

        modelo.fit(X_sub, y_sub)
        pred_treino = modelo.predict(X_sub)
        pred_validacao = modelo.predict(validacao[variaveis])
        linhas.append(
            {
                "ano_final_treino": ano_final,
                "n_treino": len(subtreino),
                "RMSE_treino": np.sqrt(mean_squared_error(y_sub, pred_treino)),
                "RMSE_validacao": np.sqrt(mean_squared_error(validacao[ALVO], pred_validacao)),
            }
        )

    return pd.DataFrame(linhas)


def salvar_curva_aprendizado(df_curva, titulo, nome_arquivo):
    plt.figure(figsize=(8, 5))
    plt.plot(df_curva["n_treino"], df_curva["RMSE_treino"], marker="o", label="Treino")
    plt.plot(
        df_curva["n_treino"],
        df_curva["RMSE_validacao"],
        marker="o",
        label="Validação",
    )
    plt.xlabel("Número de observações de treinamento")
    plt.ylabel("RMSE (L/vaca/ano)")
    plt.title(titulo)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PASTA_FIGURAS / nome_arquivo, dpi=300)
    plt.show()


curvas_aprendizado = {}
for configuracao in CONJUNTOS_VARIAVEIS:
    for nome_modelo, slug in [("Ridge", "ridge"), ("Random Forest", "random_forest")]:
        curva = gerar_curva_aprendizado_sklearn(configuracao, nome_modelo)
        curvas_aprendizado[(configuracao, nome_modelo)] = curva
        curva.to_csv(
            PASTA_RESULTADOS / f"curva_{slug}_{configuracao}.csv", index=False
        )
        salvar_curva_aprendizado(
            curva,
            f"Curva de aprendizado — {nome_modelo} — {configuracao}",
            f"curva_{slug}_{configuracao}.png",
        )

In [ ]:
# Curva de referência da persistência.
linhas_persistencia = []
for ano_final in CORTES_CURVA_APRENDIZADO:
    subtreino = treino.loc[treino["ano"] <= ano_final]
    linhas_persistencia.append(
        {
            "ano_final_treino": ano_final,
            "n_treino": len(subtreino),
            "RMSE_treino": np.sqrt(
                mean_squared_error(
                    subtreino[ALVO], subtreino["produtividade_l_vaca_ano"]
                )
            ),
            "RMSE_validacao": metricas_persistencia_validacao["RMSE"],
        }
    )
curva_persistencia = pd.DataFrame(linhas_persistencia)
curva_persistencia.to_csv(PASTA_RESULTADOS / "curva_persistencia.csv", index=False)
salvar_curva_aprendizado(
    curva_persistencia,
    "Curva de referência — Persistência",
    "curva_persistencia.png",
)

In [ ]:
# Curvas nativas do XGBoost.
for configuracao in CONJUNTOS_VARIAVEIS:
    modelo = objetos_validacao[(configuracao, "XGBoost")]["modelo"]
    historico = modelo.evals_result()
    rmse_treino = historico["validation_0"]["rmse"]
    rmse_validacao = historico["validation_1"]["rmse"]
    curva = pd.DataFrame(
        {
            "rodada": np.arange(1, len(rmse_treino) + 1),
            "RMSE_treino": rmse_treino,
            "RMSE_validacao": rmse_validacao,
        }
    )
    curva.to_csv(PASTA_RESULTADOS / f"curva_xgboost_{configuracao}.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(curva["rodada"], curva["RMSE_treino"], label="Treino")
    plt.plot(curva["rodada"], curva["RMSE_validacao"], label="Validação")
    plt.xlabel("Rodada de boosting")
    plt.ylabel("RMSE (L/vaca/ano)")
    plt.title(f"Curva de treinamento — XGBoost — {configuracao}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PASTA_FIGURAS / f"curva_xgboost_{configuracao}.png", dpi=300)
    plt.show()

In [ ]:
# Curvas nativas da MLP.
for configuracao in CONJUNTOS_VARIAVEIS:
    objeto_mlp_validacao = objetos_validacao[(configuracao, "MLP")]
    historico = objeto_mlp_validacao["historico"]
    escala_alvo = float(objeto_mlp_validacao["padronizador_alvo"].scale_[0])
    curva = pd.DataFrame(
        {
            "epoca": np.arange(1, len(historico["loss"]) + 1),
            "RMSE_treino": np.asarray(historico["rmse"]) * escala_alvo,
            "RMSE_validacao": np.asarray(historico["val_rmse"]) * escala_alvo,
        }
    )
    curva.to_csv(PASTA_RESULTADOS / f"curva_mlp_{configuracao}.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(curva["epoca"], curva["RMSE_treino"], label="Treino")
    plt.plot(curva["epoca"], curva["RMSE_validacao"], label="Validação")
    plt.xlabel("Época")
    plt.ylabel("RMSE (L/vaca/ano)")
    plt.title(f"Curva de treinamento — MLP — {configuracao}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PASTA_FIGURAS / f"curva_mlp_{configuracao}.png", dpi=300)
    plt.show()

## 11. Reajuste com treino + validação e avaliação final

Após a seleção dos hiperparâmetros, os modelos são reajustados com a união dos conjuntos de treinamento e validação. O teste é utilizado uma única vez para a avaliação final.

In [ ]:
treino_validacao = pd.concat([treino, validacao], ignore_index=True)
resultados_teste = []
predicoes_teste = []
modelos_finais = {}

# Persistência no teste.
pred_persistencia = teste["produtividade_l_vaca_ano"].to_numpy()
metricas_persistencia_teste = calcular_metricas(teste[ALVO], pred_persistencia)
resultados_teste.append(
    {
        "configuracao": "referencia",
        "modelo": "Persistência",
        **metricas_persistencia_teste,
        "melhoria_rmse_percentual": 0.0,
        "parametros": "produtividade(t+1) = produtividade(t)",
    }
)
predicoes_teste.append(
    teste[COLUNAS_IDENTIFICACAO + ["regiao", "produtividade_l_vaca_ano", ALVO]]
    .assign(configuracao="referencia", modelo="Persistência", predicao=pred_persistencia)
)

for configuracao, variaveis in CONJUNTOS_VARIAVEIS.items():
    print(f"\nReajuste final: {configuracao}")
    X_tv = treino_validacao[variaveis]
    y_tv = treino_validacao[ALVO]
    X_teste = teste[variaveis]
    y_teste = teste[ALVO]

    # Ridge
    alpha = melhores_parametros[(configuracao, "Ridge")]["alpha"]
    modelo_ridge = criar_pipeline_ridge(alpha)
    modelo_ridge.fit(X_tv, y_tv)
    pred = modelo_ridge.predict(X_teste)
    metricas = calcular_metricas(y_teste, pred)
    resultados_teste.append(
        {
            "configuracao": configuracao,
            "modelo": "Ridge",
            **metricas,
            "melhoria_rmse_percentual": 100
            * (metricas_persistencia_teste["RMSE"] - metricas["RMSE"])
            / metricas_persistencia_teste["RMSE"],
            "parametros": json.dumps({"alpha": alpha}),
        }
    )
    modelos_finais[(configuracao, "Ridge")] = modelo_ridge
    predicoes_teste.append(
        teste[COLUNAS_IDENTIFICACAO + ["regiao", "produtividade_l_vaca_ano", ALVO]]
        .assign(configuracao=configuracao, modelo="Ridge", predicao=pred)
    )

    # Random Forest
    parametros_rf = melhores_parametros[(configuracao, "Random Forest")]
    modelo_rf = criar_pipeline_rf(parametros_rf)
    modelo_rf.fit(X_tv, y_tv)
    pred = modelo_rf.predict(X_teste)
    metricas = calcular_metricas(y_teste, pred)
    resultados_teste.append(
        {
            "configuracao": configuracao,
            "modelo": "Random Forest",
            **metricas,
            "melhoria_rmse_percentual": 100
            * (metricas_persistencia_teste["RMSE"] - metricas["RMSE"])
            / metricas_persistencia_teste["RMSE"],
            "parametros": json.dumps(parametros_rf),
        }
    )
    modelos_finais[(configuracao, "Random Forest")] = modelo_rf
    predicoes_teste.append(
        teste[COLUNAS_IDENTIFICACAO + ["regiao", "produtividade_l_vaca_ano", ALVO]]
        .assign(configuracao=configuracao, modelo="Random Forest", predicao=pred)
    )

    # XGBoost
    parametros_xgb = melhores_parametros[(configuracao, "XGBoost")].copy()
    n_estimators = parametros_xgb.pop("n_estimators")
    imputador_xgb = SimpleImputer(strategy="median")
    X_tv_xgb = imputador_xgb.fit_transform(X_tv)
    X_teste_xgb = imputador_xgb.transform(X_teste)
    modelo_xgb = criar_xgb(
        parametros_xgb,
        n_estimators=n_estimators,
        early_stopping=False,
    )
    modelo_xgb.fit(X_tv_xgb, y_tv, verbose=False)
    pred = modelo_xgb.predict(X_teste_xgb)
    metricas = calcular_metricas(y_teste, pred)
    resultados_teste.append(
        {
            "configuracao": configuracao,
            "modelo": "XGBoost",
            **metricas,
            "melhoria_rmse_percentual": 100
            * (metricas_persistencia_teste["RMSE"] - metricas["RMSE"])
            / metricas_persistencia_teste["RMSE"],
            "parametros": json.dumps(
                {**parametros_xgb, "n_estimators": n_estimators}
            ),
        }
    )
    modelos_finais[(configuracao, "XGBoost")] = {
        "imputador": imputador_xgb,
        "modelo": modelo_xgb,
    }
    predicoes_teste.append(
        teste[COLUNAS_IDENTIFICACAO + ["regiao", "produtividade_l_vaca_ano", ALVO]]
        .assign(configuracao=configuracao, modelo="XGBoost", predicao=pred)
    )

    # MLP
    parametros_mlp = melhores_parametros[(configuracao, "MLP")]
    imputador_mlp = SimpleImputer(strategy="median")
    padronizador_mlp = StandardScaler()
    padronizador_alvo_mlp = StandardScaler()
    X_tv_mlp = padronizador_mlp.fit_transform(imputador_mlp.fit_transform(X_tv))
    X_teste_mlp = padronizador_mlp.transform(imputador_mlp.transform(X_teste))
    y_tv_mlp = padronizador_alvo_mlp.fit_transform(y_tv.to_numpy().reshape(-1, 1)).ravel()
    modelo_mlp = construir_mlp(
        n_entradas=X_tv_mlp.shape[1],
        camadas_ocultas=parametros_mlp["camadas"],
        dropout=parametros_mlp["dropout"],
        taxa_aprendizado=parametros_mlp["taxa_aprendizado"],
    )
    modelo_mlp.fit(
        X_tv_mlp,
        y_tv_mlp,
        epochs=parametros_mlp["epocas"],
        batch_size=TAMANHO_LOTE,
        verbose=0,
        shuffle=True,
    )
    pred_padronizada = modelo_mlp.predict(X_teste_mlp, verbose=0).ravel()
    pred = padronizador_alvo_mlp.inverse_transform(pred_padronizada.reshape(-1, 1)).ravel()
    metricas = calcular_metricas(y_teste, pred)
    resultados_teste.append(
        {
            "configuracao": configuracao,
            "modelo": "MLP",
            **metricas,
            "melhoria_rmse_percentual": 100
            * (metricas_persistencia_teste["RMSE"] - metricas["RMSE"])
            / metricas_persistencia_teste["RMSE"],
            "parametros": json.dumps(
                {
                    "camadas": list(parametros_mlp["camadas"]),
                    "dropout": parametros_mlp["dropout"],
                    "taxa_aprendizado": parametros_mlp["taxa_aprendizado"],
                    "epocas": parametros_mlp["epocas"],
                }
            ),
        }
    )
    modelos_finais[(configuracao, "MLP")] = {
        "imputador": imputador_mlp,
        "padronizador": padronizador_mlp,
        "padronizador_alvo": padronizador_alvo_mlp,
        "modelo": modelo_mlp,
    }
    predicoes_teste.append(
        teste[COLUNAS_IDENTIFICACAO + ["regiao", "produtividade_l_vaca_ano", ALVO]]
        .assign(configuracao=configuracao, modelo="MLP", predicao=pred)
    )

    gc.collect()

resultados_teste = pd.DataFrame(resultados_teste).sort_values("RMSE")
predicoes_teste = pd.concat(predicoes_teste, ignore_index=True)
predicoes_teste["erro"] = predicoes_teste["predicao"] - predicoes_teste[ALVO]
predicoes_teste["erro_absoluto"] = predicoes_teste["erro"].abs()

display(resultados_teste)
resultados_teste.to_csv(PASTA_RESULTADOS / "resultados_teste.csv", index=False)
predicoes_teste.to_csv(PASTA_RESULTADOS / "predicoes_teste.csv", index=False)

## 12. Comparação dos resultados

In [ ]:
# RMSE dos modelos e configurações.
plt.figure(figsize=(10, 6))
tabela_plot = resultados_teste.copy()
tabela_plot["rotulo"] = tabela_plot["modelo"] + " — " + tabela_plot["configuracao"]
tabela_plot = tabela_plot.sort_values("RMSE", ascending=True)
plt.barh(tabela_plot["rotulo"], tabela_plot["RMSE"])
plt.xlabel("RMSE (L/vaca/ano)")
plt.ylabel("")
plt.title("Desempenho no conjunto de teste")
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "comparacao_rmse_teste.png", dpi=300)
plt.show()

In [ ]:
# Ganho incremental dos conjuntos de informação em cada algoritmo.
comparacao_configuracoes = resultados_teste.loc[
    resultados_teste["configuracao"].isin(CONJUNTOS_VARIAVEIS.keys())
].pivot(index="modelo", columns="configuracao", values="RMSE")

for coluna in ["historico", "pecuario", "integrado"]:
    if coluna not in comparacao_configuracoes.columns:
        comparacao_configuracoes[coluna] = np.nan

comparacao_configuracoes["ganho_pecuario_vs_historico_pct"] = 100 * (
    comparacao_configuracoes["historico"] - comparacao_configuracoes["pecuario"]
) / comparacao_configuracoes["historico"]
comparacao_configuracoes["ganho_integrado_vs_pecuario_pct"] = 100 * (
    comparacao_configuracoes["pecuario"] - comparacao_configuracoes["integrado"]
) / comparacao_configuracoes["pecuario"]
comparacao_configuracoes["ganho_integrado_vs_historico_pct"] = 100 * (
    comparacao_configuracoes["historico"] - comparacao_configuracoes["integrado"]
) / comparacao_configuracoes["historico"]

display(comparacao_configuracoes)
comparacao_configuracoes.to_csv(
    PASTA_RESULTADOS / "ganhos_por_conjunto_de_informacao.csv"
)

In [ ]:
# Desempenho por ano-alvo, região e faixa inicial de produtividade.
limites_faixas = treino["produtividade_l_vaca_ano"].quantile([1 / 3, 2 / 3]).to_numpy()
predicoes_teste["faixa_produtividade_inicial"] = pd.cut(
    predicoes_teste["produtividade_l_vaca_ano"],
    bins=[-np.inf, limites_faixas[0], limites_faixas[1], np.inf],
    labels=["Baixa", "Intermediária", "Alta"],
)


def metricas_por_grupo(df, colunas_grupo):
    linhas = []
    for chaves, grupo in df.groupby(colunas_grupo, observed=True):
        if not isinstance(chaves, tuple):
            chaves = (chaves,)
        metricas = calcular_metricas(grupo[ALVO], grupo["predicao"])
        linha = dict(zip(colunas_grupo, chaves))
        linha.update(metricas)
        linha["n"] = len(grupo)
        linhas.append(linha)
    return pd.DataFrame(linhas)


metricas_ano = metricas_por_grupo(
    predicoes_teste, ["configuracao", "modelo", "ano_alvo"]
)
metricas_regiao = metricas_por_grupo(
    predicoes_teste, ["configuracao", "modelo", "regiao"]
)
metricas_faixa = metricas_por_grupo(
    predicoes_teste, ["configuracao", "modelo", "faixa_produtividade_inicial"]
)

metricas_ano.to_csv(PASTA_RESULTADOS / "metricas_por_ano.csv", index=False)
metricas_regiao.to_csv(PASTA_RESULTADOS / "metricas_por_regiao.csv", index=False)
metricas_faixa.to_csv(PASTA_RESULTADOS / "metricas_por_faixa.csv", index=False)

display(metricas_ano.sort_values(["modelo", "configuracao", "ano_alvo"]).head(20))

In [ ]:
# Observado versus predito para o melhor modelo do teste.
melhor_linha = resultados_teste.iloc[0]
melhor_configuracao = melhor_linha["configuracao"]
melhor_modelo = melhor_linha["modelo"]
melhores_predicoes = predicoes_teste.loc[
    (predicoes_teste["configuracao"] == melhor_configuracao)
    & (predicoes_teste["modelo"] == melhor_modelo)
]

plt.figure(figsize=(6, 6))
plt.scatter(
    melhores_predicoes[ALVO],
    melhores_predicoes["predicao"],
    alpha=0.25,
    s=10,
)
limite = max(melhores_predicoes[ALVO].max(), melhores_predicoes["predicao"].max())
plt.plot([0, limite], [0, limite], linestyle="--")
plt.xlabel("Produtividade observada (L/vaca/ano)")
plt.ylabel("Produtividade predita (L/vaca/ano)")
plt.title(f"Observado versus predito — {melhor_modelo} — {melhor_configuracao}")
plt.tight_layout()
plt.savefig(PASTA_FIGURAS / "observado_versus_predito_melhor_modelo.png", dpi=300)
plt.show()

## 13. Interpretação dos modelos

A interpretação é realizada na configuração integrada, que contém as 21 variáveis. Para a Ridge são apresentados os coeficientes padronizados. Random Forest e XGBoost são avaliados por importância por permutação. Na MLP é utilizada a extensão do método de pesos de conexão de Olden para múltiplas camadas.

In [ ]:
CONFIGURACAO_INTERPRETACAO = "integrado"
VARIAVEIS_INTERPRETACAO = CONJUNTOS_VARIAVEIS[CONFIGURACAO_INTERPRETACAO]
importancias = []

# Ridge: coeficientes após imputação e padronização.
modelo_ridge = modelos_finais[(CONFIGURACAO_INTERPRETACAO, "Ridge")]
coeficientes = modelo_ridge.named_steps["modelo"].coef_
for variavel, valor in zip(VARIAVEIS_INTERPRETACAO, coeficientes):
    importancias.append(
        {
            "modelo": "Ridge",
            "variavel": variavel,
            "importancia": valor,
            "importancia_absoluta": abs(valor),
            "metodo": "coeficiente padronizado",
        }
    )

# Amostra fixa para limitar o custo da importância por permutação.
amostra_interpretacao = teste.sample(
    n=min(20000, len(teste)), random_state=SEMENTE
)
X_interpretacao = amostra_interpretacao[VARIAVEIS_INTERPRETACAO]
y_interpretacao = amostra_interpretacao[ALVO]

# Random Forest.
modelo_rf = modelos_finais[(CONFIGURACAO_INTERPRETACAO, "Random Forest")]
perm_rf = permutation_importance(
    modelo_rf,
    X_interpretacao,
    y_interpretacao,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=SEMENTE,
    n_jobs=-1,
)
for variavel, valor in zip(VARIAVEIS_INTERPRETACAO, perm_rf.importances_mean):
    importancias.append(
        {
            "modelo": "Random Forest",
            "variavel": variavel,
            "importancia": valor,
            "importancia_absoluta": abs(valor),
            "metodo": "permutação",
        }
    )

# XGBoost.
objeto_xgb = modelos_finais[(CONFIGURACAO_INTERPRETACAO, "XGBoost")]
X_interpretacao_xgb = objeto_xgb["imputador"].transform(X_interpretacao)
perm_xgb = permutation_importance(
    objeto_xgb["modelo"],
    X_interpretacao_xgb,
    y_interpretacao,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=SEMENTE,
    n_jobs=-1,
)
for variavel, valor in zip(VARIAVEIS_INTERPRETACAO, perm_xgb.importances_mean):
    importancias.append(
        {
            "modelo": "XGBoost",
            "variavel": variavel,
            "importancia": valor,
            "importancia_absoluta": abs(valor),
            "metodo": "permutação",
        }
    )

# MLP: produto das matrizes de pesos entre a entrada e a saída.
objeto_mlp = modelos_finais[(CONFIGURACAO_INTERPRETACAO, "MLP")]
matrizes_pesos = [
    camada.get_weights()[0]
    for camada in objeto_mlp["modelo"].layers
    if isinstance(camada, layers.Dense)
]
contribuicao_olden = matrizes_pesos[0]
for matriz in matrizes_pesos[1:]:
    contribuicao_olden = contribuicao_olden @ matriz
contribuicao_olden = contribuicao_olden.ravel()

for variavel, valor in zip(VARIAVEIS_INTERPRETACAO, contribuicao_olden):
    importancias.append(
        {
            "modelo": "MLP",
            "variavel": variavel,
            "importancia": valor,
            "importancia_absoluta": abs(valor),
            "metodo": "Olden generalizado",
        }
    )

importancias = pd.DataFrame(importancias)
importancias["importancia_relativa_pct"] = importancias.groupby("modelo")[
    "importancia_absoluta"
].transform(lambda s: 100 * s / s.sum())
importancias.to_csv(PASTA_RESULTADOS / "importancia_variaveis.csv", index=False)
display(importancias.sort_values(["modelo", "importancia_absoluta"], ascending=[True, False]))

In [ ]:
for nome_modelo in ["Ridge", "Random Forest", "XGBoost", "MLP"]:
    dados = (
        importancias.loc[importancias["modelo"] == nome_modelo]
        .nlargest(15, "importancia_absoluta")
        .sort_values("importancia_absoluta")
    )
    plt.figure(figsize=(9, 6))
    plt.barh(dados["variavel"], dados["importancia_relativa_pct"])
    plt.xlabel("Importância relativa (%)")
    plt.ylabel("")
    plt.title(f"Importância das variáveis — {nome_modelo} — integrado")
    plt.tight_layout()
    slug = nome_modelo.lower().replace(" ", "_")
    plt.savefig(PASTA_FIGURAS / f"importancia_{slug}.png", dpi=300)
    plt.show()

## 14. Salvamento dos modelos, tabelas e pacote final

In [ ]:
# Salva a base de modelagem e as especificações do experimento.
colunas_base_exportacao = (
    COLUNAS_IDENTIFICACAO
    + ["regiao", "produtividade_l_vaca_ano", ALVO]
    + CONJUNTOS_VARIAVEIS["integrado"]
)
colunas_base_exportacao = list(dict.fromkeys(colunas_base_exportacao))
base_modelagem[colunas_base_exportacao].to_parquet(
    PASTA_RESULTADOS / "base_modelagem.parquet", index=False
)

configuracao_experimento = {
    "semente": SEMENTE,
    "intervalos": INTERVALOS,
    "conjuntos_variaveis": CONJUNTOS_VARIAVEIS,
    "melhores_parametros": {
        f"{configuracao}__{modelo}": {
            chave: list(valor) if isinstance(valor, tuple) else valor
            for chave, valor in parametros.items()
        }
        for (configuracao, modelo), parametros in melhores_parametros.items()
    },
    "versoes": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "tensorflow": tf.__version__,
    },
}
with open(PASTA_RESULTADOS / "configuracao_experimento.json", "w", encoding="utf-8") as arquivo:
    json.dump(configuracao_experimento, arquivo, ensure_ascii=False, indent=2)

# Modelos scikit-learn e XGBoost.
for (configuracao, nome_modelo), objeto in modelos_finais.items():
    slug_modelo = nome_modelo.lower().replace(" ", "_")
    if nome_modelo == "MLP":
        joblib.dump(
            {
                "imputador": objeto["imputador"],
                "padronizador": objeto["padronizador"],
                "padronizador_alvo": objeto["padronizador_alvo"],
                "variaveis": CONJUNTOS_VARIAVEIS[configuracao],
            },
            PASTA_MODELOS / f"preprocessamento_mlp_{configuracao}.joblib",
        )
        objeto["modelo"].save(PASTA_MODELOS / f"mlp_{configuracao}.keras")
    else:
        joblib.dump(
            {
                "objeto": objeto,
                "variaveis": CONJUNTOS_VARIAVEIS[configuracao],
            },
            PASTA_MODELOS / f"{slug_modelo}_{configuracao}.joblib",
        )

# Consolida as principais tabelas em uma planilha.
with pd.ExcelWriter(PASTA_RESULTADOS / "resultados_consolidados.xlsx") as writer:
    resultados_validacao.to_excel(writer, sheet_name="validacao", index=False)
    resultados_teste.to_excel(writer, sheet_name="teste", index=False)
    comparacao_configuracoes.to_excel(writer, sheet_name="ganhos_informacao")
    metricas_ano.to_excel(writer, sheet_name="por_ano", index=False)
    metricas_regiao.to_excel(writer, sheet_name="por_regiao", index=False)
    metricas_faixa.to_excel(writer, sheet_name="por_faixa", index=False)
    importancias.to_excel(writer, sheet_name="importancias", index=False)
    resumo_qualidade.to_excel(writer, sheet_name="qualidade_dados", index=False)

# Cria um arquivo compactado com todos os resultados.
arquivo_zip_resultados = shutil.make_archive(
    str(PASTA_PROJETO / "resultados_produtividade_leiteira_ml"),
    "zip",
    root_dir=PASTA_RESULTADOS,
)

print("Execução concluída.")
print("Pasta de resultados:", PASTA_RESULTADOS)
print("Arquivo compactado:", arquivo_zip_resultados)